In [1]:
import os
os.chdir("/home/ubuntu/work/dahyeon/backend")


In [2]:
import os
os.getcwd()

'/home/ubuntu/work/dahyeon/backend'

In [13]:
from app.modules.RAG.retriever import (
  full_bm25_with_onboarding,
  full_dense_with_onboarding,
  full_hybrid_with_onboarding,
  make_onboarding_result)

sample_result = {
          "keyword_query": ["한국소설", "에세이", "도시생활", "힐링", "문학적"],
          "semantic_query": "도시 생활에 지쳤을 때 힐링이 되는 문학적인 한국 작가 소설·에세이",
          "filters": {
            "cate_depth1": ["소설", "시/에세이"],
            "custom_constraints": ["한국 작가 우선", "문학적인 문체"]
          },
          "constraints": {},
          "score_boost": { "cate_depth2": ["한국소설", "테마에세이"], "subject": ["도시생활", "힐링"] },
          "session_signals": {
            "mood": ["negative_stressed", "recovery_relax"],
            "purpose": "재미",
            "reading_level": "medium"
          },
          "onboarding_signals": { "length_soft": { "operator": "lte", "value": 250 } }
        }
    
# results = full_bm25_with_onboarding(sample_result, size=40)

# for r in results:
#     print(
#         r["rank"],
#         r.get("title"),
#         r.get("large_cate"),
#         r.get("score"),
#         r.get("main_score"),
#         r.get("onboarding_score"),
#         r.get("disliked_penalty"),
#         r.get("disliked_similarity")
    
re1_bm25 = full_bm25_with_onboarding(sample_result, size=40)
re1_dense = full_dense_with_onboarding(sample_result, size=40)
re1_hybrid = full_hybrid_with_onboarding(sample_result, size=40)

/home/ubuntu/anaconda3/envs/venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/ubuntu/anaconda3/envs/venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/ubuntu/anaconda3/envs/venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/ubuntu/anacon

In [15]:
import json

def deduplicate_results(results_list, key="isbn"):
    merged = []
    seen = set()

    for results in results_list:
        for item in results:

            item_key = item.get(key)

            if item_key in seen:
                continue

            seen.add(item_key)
            merged.append(item)

    return merged


# =========================
# full 결과 통합
# =========================

full_merged = deduplicate_results([
    re1_bm25,
    re1_dense,
    re1_hybrid
])

full_result_json = {
    "query": sample_result,
    "results": full_merged
}

#with open("full_results.json", "w", encoding="utf-8") as f:
#    json.dump(full_result_json, f, ensure_ascii=False, indent=2)

def simplify_item(item, source_name):
    return {
        "isbn": item.get("isbn"),
        "title": item.get("title"),
        "author": item.get("author"),
        "publisher": item.get("publisher"),
        "publish_date": item.get("publish_date"),
        "page": item.get("page"),

        "large_cate": item.get("large_cate"),
        "mid_cate": item.get("mid_cate"),
        "small_cate": item.get("small_cate"),

        "book_intro": item.get("book_intro"),
        "book_index": item.get("book_index"),

        "review_count": item.get("review_count"),
        "review_text": item.get("review_text"),

        # "retrieval_source": source_name,
        # "retrieval_rank": item.get("rank"),
        # "retrieval_score": item.get("score")
    }


def merge_with_sources(result_groups, key="isbn"):
    merged = {}

    for source_name, results in result_groups.items():
        for item in results:
            item_key = item.get(key)

            if item_key is None:
                continue

            simplified = simplify_item(item, source_name)

            if item_key not in merged:
                base = {
                    k: v for k, v in simplified.items()
                    if not k.startswith("retrieval_")
                }

                base["retrieval_info"] = []
                merged[item_key] = base

            merged[item_key]["retrieval_info"].append({
                "source": source_name,
                "rank": item.get("rank"),
                "score": item.get("score")
            })

    return list(merged.values())

full_books = merge_with_sources({
    "bm25": re1_bm25,
    "dense": re1_dense,
    "hybrid": re1_hybrid
})

#with open("full_books_for_labeling.json", "w", encoding="utf-8") as f:
#   json.dump(full_books, f, ensure_ascii=False, indent=2)





In [21]:
import json
import uuid
import time
import random
import requests
from tqdm import tqdm
from app.core.config import settings

URL = "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"

CLOVA_API_KEY = settings.CLOVA_API_KEY


SYSTEM_PROMPT = """
당신은 매우 보수적으로 책 추천 적합성을 판단하는 평가자입니다.

입력으로는 다음 두 정보가 주어집니다.

1. request: 사용자의 검색 요청 전체
- keyword_query
- semantic_query
- filters
- constraints
- score_boost
- session_signals
- onboarding_signals

2. book: 추천 후보 도서 정보

[판정 기준]
- 현재 사용자 요청의 핵심 의도와 조건을 우선적으로 판단하세요.
- 사용자 요청의 모든 표현이 책 정보에 직접 등장할 필요는 없습니다.
- 제목, 소개, 목차, 카테고리, 리뷰 등을 종합하여 의미적 적합성을 판단하세요.
- 책이 제공하는 독서 경험, 분위기, 주제, 대상 독자가 사용자 요청과 충분히 일치하면 1로 판단할 수 있습니다.
- 일부 키워드만 우연히 일치하거나, 장르/주제/대상/분위기가 실제로 맞지 않으면 0으로 판단하세요.
- filters, constraints는 명시 조건으로 보고 우선 고려하세요.
- onboarding_signals와 session_signals는 사용자 선호 신호이므로 보조적으로 반영하세요.
- 단, soft 조건은 약간 벗어나도 핵심 요청 적합성이 높으면 1로 판단할 수 있습니다.
- 정보가 부족하여 핵심 조건 충족 여부를 판단할 수 없으면 0으로 분류하세요.
- 요청과 직접 관련 없는 책이 단순히 키워드만 포함한 경우에는 0으로 분류하세요.

출력 형식:
{
  "label": 0 또는 1
}
"""


def make_headers():
    return {
        "Authorization": f"Bearer {CLOVA_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "text/event-stream"
    }


def parse_hcx_response(text):

    # =========================
    # 1. SSE 형식 처리
    # =========================
    if "event:" in text:

        last_data = None

        for block in text.split("\n\n"):

            lines = block.strip().splitlines()

            event_type = None
            data_text = None

            for line in lines:

                if line.startswith("event:"):
                    event_type = line.replace("event:", "").strip()

                elif line.startswith("data:"):
                    data_text = line.replace("data:", "").strip()

            if event_type == "result" and data_text:
                last_data = data_text

        if last_data is not None:
            data = json.loads(last_data)
            return data["message"]["content"]

    # =========================
    # 2. 일반 JSON 응답 처리
    # =========================
    try:
        data = json.loads(text)

        if "result" in data:
            return data["result"]["message"]["content"]

        if "message" in data:
            return data["message"]["content"]

    except:
        pass

    raise ValueError(f"응답 파싱 실패:\n{text[:1000]}")


def extract_label(llm_text):
    try:
        obj = json.loads(llm_text)
        label = int(obj.get("label", 0))
        return 1 if label == 1 else 0
    except Exception:
        # 파싱 실패하면 보수적으로 0
        return 0


def make_user_prompt(request_data, book):
    payload = {
        "request": request_data,
        "book": book
    }

    return json.dumps(payload, ensure_ascii=False, indent=2)


def label_one_book(request_data, book, max_retries=5):
    user_prompt = make_user_prompt(request_data, book)

    body = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        "topP": 0.1,
        "topK": 0,
        "max_tokens": 100,
        "temperature": 0.0,
        "repetitionPenalty": 1.0,
        "includeAiFilters": False,
        "seed": 42
    }

    for attempt in range(max_retries):
        try:
            res = requests.post(
                URL,
                headers=make_headers(),
                json=body,
                timeout=60
            )

            if res.status_code == 200:
                llm_text = parse_hcx_response(res.text)
                label = extract_label(llm_text)

                return {
                    **book,
                    "llm_label": label,
                    "llm_raw_output": llm_text
                }

            if res.status_code in [429, 500, 502, 503, 504]:
                raise Exception(f"{res.status_code} / {res.text[:500]}")

            raise Exception(f"{res.status_code} / {res.text[:500]}")

        except Exception as e:
            if attempt == max_retries - 1:
                return {
                    **book,
                    "llm_label": 0,
                    "llm_error": str(e)
                }

            wait = min(30, (2 ** attempt) + random.uniform(0, 1))
            print(f"[재시도 {attempt+1}/{max_retries}] {wait:.1f}초 대기 → {e}")
            time.sleep(wait)

labeled_full_books = []

for book in tqdm(full_books):
    labeled = label_one_book(
        request_data=sample_result,
        book=book
    )

    labeled_full_books.append(labeled)

with open("full_books_labeled.json", "w", encoding="utf-8") as f:
    json.dump(labeled_full_books, f, ensure_ascii=False, indent=2)

  0%|          | 0/86 [00:00<?, ?it/s]

100%|██████████| 86/86 [14:08<00:00,  9.87s/it]
